# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
#%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [3]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x: (batch, in_dim) -> LoRA update (batch, out_dim)
        # Compute low-rank weight delta = (alpha / r) * A @ B
        rank = self.A.shape[1]
        delta_w = (self.A @ self.B) * (self.alpha / rank)  # (in_dim, out_dim)
        x = x @ delta_w
        return x


# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 4
out_dim = 3
rank = 2
alpha = 8

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(2, in_dim)  # batch=2

print(x)
print(layer)
print("Original output:", layer(x))


Using device: cpu
tensor([[ 0.3239, -0.1085,  0.2103, -0.3908],
        [ 0.2350,  0.6653,  0.3528,  0.9728]])
LoRALayer()
Original output: tensor([[0., 0., 0.],
        [0., 0., 0.]], grad_fn=<MmBackward0>)


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [4]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # Base linear projection + LoRA correction
        return self.linear(x) + self.lora(x)


# Simple test
base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)

print("LinearWithLoRA output:", layer_lora_1(x))


LinearWithLoRA output: tensor([[ 0.1220, -0.1925, -0.2738],
        [ 0.2311, -0.2710,  0.1665]], grad_fn=<AddBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [ ]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        return TODO

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = TODO  # wrap `base_linear` with rank/alpha values
print("LinearWithLoRA output:", layer_lora_1(x))

# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [5]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features: int, num_classes: int):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)


num_features = 10
num_classes = 4

single_net = SingleLayerNet(num_features=num_features, num_classes=num_classes)
sample_input = torch.randn(16, num_features)

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Replace linear block with LinearWithLoRA
single_net.layer = LinearWithLoRA(single_net.layer, rank=4, alpha=8)

with torch.no_grad():
    lora_output = single_net(sample_input)

print("Outputs match before training?",
      torch.allclose(baseline_output, lora_output, atol=1e-6))


Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [6]:
# This LoRA code is equivalent to LinearWithLoRA but merges weights
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # Combine LoRA matrices
        lora = self.lora.A @ self.lora.B  # (in_dim, out_dim)
        rank = self.lora.A.shape[1]
        # Merge original weights + scaled LoRA correction
        combined_weight = self.linear.weight + (self.lora.alpha / rank) * lora.T
        return F.linear(x, combined_weight, self.linear.bias)


layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=rank, alpha=alpha)
print("Merged LoRA output:", layer_lora_2(x))


Merged LoRA output: tensor([[ 0.1220, -0.1925, -0.2738],
        [ 0.2311, -0.2710,  0.1665]], grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [8]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        x = self.layers(x)
        return x




In [9]:
# Architecture
num_features = 28 * 28
num_hidden_1 = 256
num_hidden_2 = 128
num_classes = 10

# Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 1e-3
num_epochs = 3  # keep it small for Colab

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)


cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [11]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

# Note: transforms.ToTensor() scales input images to 0-1 range
train_dataset = datasets.MNIST(
    root="data", train=True, transform=transforms.ToTensor(), download=True
)

test_dataset = datasets.MNIST(
    root="data", train=False, transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print("Image batch dimensions:", images.shape)
    print("Image label dimensions:", labels.shape)
    break


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [12]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    return 100 * correct_pred / num_examples


## Training

In [14]:
import time

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)

            # forward and back propagation
            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            # update model parameters
            optimizer.step()

            # logging
            if not batch_idx % 400:
                print(
                    "Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f"
                    % (epoch + 1, num_epochs, batch_idx, len(train_loader), loss)
                )

        with torch.set_grad_enabled(False):
            print(
                "Epoch: %03d/%03d training accuracy: %.2f%%"
                % (epoch + 1, num_epochs, compute_accuracy(model, train_loader, device))
            )

        print("Time elapsed: %.2f min" % ((time.time() - start_time) / 60))
    print("Total Training Time: %.2f min" % ((time.time() - start_time) / 60))


In [15]:

train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f"Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%")


Epoch: 001/003 | Batch 000/938 | Loss: 0.1110
Epoch: 001/003 | Batch 400/938 | Loss: 0.1878
Epoch: 001/003 | Batch 800/938 | Loss: 0.2870
Epoch: 001/003 training accuracy: 97.88%
Time elapsed: 0.36 min
Epoch: 002/003 | Batch 000/938 | Loss: 0.0876
Epoch: 002/003 | Batch 400/938 | Loss: 0.1504
Epoch: 002/003 | Batch 800/938 | Loss: 0.0467
Epoch: 002/003 training accuracy: 98.65%
Time elapsed: 0.69 min
Epoch: 003/003 | Batch 000/938 | Loss: 0.0110
Epoch: 003/003 | Batch 400/938 | Loss: 0.1138
Epoch: 003/003 | Batch 800/938 | Loss: 0.0734
Epoch: 003/003 training accuracy: 98.77%
Time elapsed: 1.03 min
Total Training Time: 1.03 min
Test accuracy: 97.77%


# Replacing Linear with LoRA Layers

In [16]:
import copy

model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f"Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")


MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=256, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=256, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model:97.77%
Test accuracy LoRA model:97.77%


## Freezing the Original Linear Layers

In [18]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            # recursively freeze linear layers in children modules
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f"{name}: {param.requires_grad}")



layers.0.linear.weight: False
layers.0.linear.bias: False
layers.0.lora.A: True
layers.0.lora.B: True
layers.2.linear.weight: False
layers.2.linear.bias: False
layers.2.lora.A: True
layers.2.lora.B: True
layers.4.linear.weight: False
layers.4.linear.bias: False
layers.4.lora.A: True
layers.4.lora.B: True


In [ ]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f"Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

print(f"Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")


Epoch: 001/003 | Batch 000/938 | Loss: 0.0352
Epoch: 001/003 | Batch 400/938 | Loss: 0.0189
Epoch: 001/003 | Batch 800/938 | Loss: 0.0061
